In [1]:
import torch
import os
os.chdir('../')

In [2]:
from scipy import linalg
import numpy as np
import os, torch
from tqdm import tqdm
from reports.util import load_config


@torch.no_grad()
def _trace_sqrtm_product(C1: torch.Tensor, C2: torch.Tensor) -> torch.Tensor:
    # Tr sqrtm(C1 @ C2) = Tr sqrt( C1^{1/2} C2 C1^{1/2} )
    s, U = torch.linalg.eigh(C1)                 # C1 = U diag(s) U^T
    s = s.clamp_min(0)
    C1h = (U * s.sqrt()) @ U.t()                 # C1^{1/2}
    M   = C1h @ C2 @ C1h
    w   = torch.linalg.eigvalsh((M + M.t()) * 0.5).clamp_min(0)
    return w.sqrt().sum()

@torch.no_grad()
def calc_fid_stats(mu1, sigma1, mu2, sigma2, eps: float = 1e-6) -> float:
    # 모두 float64 + 동일 device로 정렬
    C1 = torch.as_tensor(sigma1, dtype=torch.float64)
    device = C1.device
    C2 = torch.as_tensor(sigma2, dtype=torch.float64).to(device)
    m1 = torch.as_tensor(mu1,    dtype=torch.float64).to(device).flatten()
    m2 = torch.as_tensor(mu2,    dtype=torch.float64).to(device).flatten()

    D = m1.numel()
    I = torch.eye(D, dtype=torch.float64, device=device)

    # 대칭화 + 정칙화
    C1 = (C1 + C1.t()) * 0.5 + eps * I
    C2 = (C2 + C2.t()) * 0.5 + eps * I

    diff = m1 - m2
    tr_covmean = _trace_sqrtm_product(C1, C2)
    fid = diff.dot(diff) + torch.trace(C1) + torch.trace(C2) - 2.0 * tr_covmean
    return float(fid)

@torch.no_grad()
def calc_fid_pt_dir(pt_dir: str, mu, sigma, eps: float = 1e-6, num=100000, key="inception_feature") -> float:
    # pt_dir에서 'inception_feature'를 모아서 mu1, sigma1 추정 후 FID 계산
    X = []
    for f in tqdm(os.listdir(pt_dir)[:num]):
        if f.endswith(".pt"):
            v = torch.load(os.path.join(pt_dir, f), map_location="cpu").get(key)
            if v is not None:
                X.append(torch.as_tensor(v, dtype=torch.float64).flatten())
    if len(X) < 2:
        raise ValueError("need >=2 features")

    X   = torch.stack(X, 0)                 # [N, D]
    mu1 = X.mean(0)
    Xc  = X - mu1
    sigma1 = (Xc.t() @ Xc) / (X.shape[0] - 1)  # 불편추정

    return calc_fid_stats(mu1, sigma1, mu, sigma, eps=eps)
    #return calculate_frechet_distance(mu1, sigma1, mu, sigma, eps=eps)

def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    """Numpy implementation of the Frechet Distance.
    The Frechet distance between two multivariate Gaussians X_1 ~ N(mu_1, C_1)
    and X_2 ~ N(mu_2, C_2) is
            d^2 = ||mu_1 - mu_2||^2 + Tr(C_1 + C_2 - 2*sqrt(C_1*C_2)).

    Stable version by Dougal J. Sutherland.

    Params:
    -- mu1   : Numpy array containing the activations of a layer of the
               inception net (like returned by the function 'get_predictions')
               for generated samples.
    -- mu2   : The sample mean over activations, precalculated on an
               representative data set.
    -- sigma1: The covariance matrix over activations for generated samples.
    -- sigma2: The covariance matrix over activations, precalculated on an
               representative data set.

    Returns:
    --   : The Frechet Distance.
    """

    mu1 = np.atleast_1d(mu1)
    mu2 = np.atleast_1d(mu2)

    sigma1 = np.atleast_2d(sigma1)
    sigma2 = np.atleast_2d(sigma2)

    assert mu1.shape == mu2.shape, \
        'Training and test mean vectors have different lengths'
    assert sigma1.shape == sigma2.shape, \
        'Training and test covariances have different dimensions'

    diff = mu1 - mu2

    # Product might be almost singular
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        msg = ('fid calculation produces singular product; '
               'adding %s to diagonal of cov estimates') % eps
        print(msg)
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    # Numerical error might give slight imaginary component
    if np.iscomplexobj(covmean):
        if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
            m = np.max(np.abs(covmean.imag))
            raise ValueError('Imaginary component {}'.format(m))
        covmean = covmean.real

    tr_covmean = np.trace(covmean)

    return (diff.dot(diff) + np.trace(sigma1)
            + np.trace(sigma2) - 2 * tr_covmean)

In [3]:

pt_dirs = ['samplings/DiT/1.5/3/Dual-Solver/50000/traj_0',
            'samplings/DiT/1.5/5/Dual-Solver/50000/traj_0',
            'samplings/DiT/1.5/7/Dual-Solver/50000/traj_0',
            'samplings/DiT/1.5/9/Dual-Solver/50000/traj_0',
            ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(config.solver, config.NFE, config.CFG, fid)

100%|██████████| 50001/50001 [00:15<00:00, 3281.26it/s]


Dual-Solver 3 1.5 100.89964025857654


100%|██████████| 50001/50001 [00:14<00:00, 3518.33it/s]


Dual-Solver 5 1.5 11.591212712497509


100%|██████████| 50001/50001 [00:14<00:00, 3565.94it/s]


Dual-Solver 7 1.5 3.6604911423949034


100%|██████████| 50001/50001 [00:13<00:00, 3574.91it/s]


Dual-Solver 9 1.5 2.8414027722069477


In [3]:
pt_dirs = ['samplings/DiT/1.5/3/Euler/50000/euler_0',
            'samplings/DiT/1.5/5/Euler/50000/euler_0',
            'samplings/DiT/1.5/7/Euler/50000/euler_0',
            'samplings/DiT/1.5/9/Euler/50000/euler_0'
            ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    #data = torch.load('/dataset/dit/stats.pt')
    data = torch.load("/home/scpark/data/imagenet_feats/train_clean_stats.pt")
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(config.solver, config.NFE, config.CFG, fid)

# 100%|██████████| 50001/50001 [00:16<00:00, 3044.41it/s]
# Euler 3 1.5 89.33531979433735
# 100%|██████████| 50001/50001 [00:14<00:00, 3382.11it/s]
# Euler 5 1.5 32.913969756826646
# 100%|██████████| 50001/50001 [00:14<00:00, 3438.57it/s]
# Euler 7 1.5 13.648913646202743
# 100%|██████████| 50001/50001 [00:15<00:00, 3308.26it/s]
# Euler 9 1.5 7.429267906242671

100%|██████████| 50001/50001 [00:22<00:00, 2208.96it/s]


Euler 3 1.5 90.97341992095375


100%|██████████| 50001/50001 [00:20<00:00, 2402.82it/s]


Euler 5 1.5 33.81300015441661


100%|██████████| 50001/50001 [00:20<00:00, 2419.92it/s]


Euler 7 1.5 14.81844233751815


100%|██████████| 50001/50001 [00:20<00:00, 2433.11it/s]


Euler 9 1.5 8.834228434839417


In [7]:
pt_dirs = ['samplings/DiT/1.5/3/DPM-Solver/50000/dpm_0',
            'samplings/DiT/1.5/5/DPM-Solver/50000/dpm_0',
            'samplings/DiT/1.5/7/DPM-Solver/50000/dpm_0',
            'samplings/DiT/1.5/9/DPM-Solver/50000/dpm_0'
            ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(config.solver, config.NFE, config.CFG, fid)

# 100%|██████████| 50001/50001 [00:13<00:00, 3712.58it/s]
# DPM-Solver 3 1.5 88.46767501303668
# 100%|██████████| 50001/50001 [09:50<00:00, 84.63it/s] 
# DPM-Solver 5 1.5 22.19664350145854
# 100%|██████████| 50001/50001 [06:56<00:00, 120.00it/s]
# DPM-Solver 7 1.5 7.06965238827712
# 100%|██████████| 50001/50001 [06:58<00:00, 119.54it/s]
# DPM-Solver 9 1.5 4.438686720234841

  0%|          | 0/50001 [00:00<?, ?it/s]

100%|██████████| 50001/50001 [00:13<00:00, 3689.91it/s]


DPM-Solver 3 1.5 88.46767501303657


100%|██████████| 50001/50001 [00:13<00:00, 3618.43it/s]


DPM-Solver 5 1.5 22.19664350145888


100%|██████████| 50001/50001 [00:13<00:00, 3729.42it/s]


DPM-Solver 7 1.5 7.069652388275699


100%|██████████| 50001/50001 [00:13<00:00, 3736.39it/s]


DPM-Solver 9 1.5 4.438686720235296


In [ ]:
pt_dirs = ['samplings/DiT/bns/vec/cfg1.5_s3_k=0.5_N50000/bns_vec_0',
           'samplings/DiT/bns/vec/cfg1.5_s5_k=0.5_N50000/bns_vec_0',
           'samplings/DiT/bns/vec/cfg1.5_s7_k=0.5_N50000/bns_vec_0',
            ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(config.solver, config.NFE, config.CFG, config.k, fid)


  0%|          | 0/50001 [00:00<?, ?it/s]

100%|██████████| 50001/50001 [00:13<00:00, 3719.73it/s]


BNS-Solver_Vec 3 1.5 0.5 180.07353879545934


In [18]:
# try2
pt_dirs = ['samplings/DiT/1.5/9/BNS-Solver/50000/bns_1',
        'samplings/DiT/1.5/7/BNS-Solver/50000/bns_1',
        'samplings/DiT/1.5/5/BNS-Solver/50000/bns_1',
        'samplings/DiT/1.5/3/BNS-Solver/50000/bns_1',
        ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(config.solver, config.NFE, config.CFG, fid)


100%|██████████| 50001/50001 [00:26<00:00, 1899.18it/s]


BNS-Solver 9 1.5 3.0309820899062174


100%|██████████| 50001/50001 [00:18<00:00, 2733.82it/s]


BNS-Solver 7 1.5 4.238185304463855


100%|██████████| 50001/50001 [00:18<00:00, 2735.84it/s]


BNS-Solver 5 1.5 14.184996013051318


100%|██████████| 50001/50001 [00:18<00:00, 2760.27it/s]


BNS-Solver 3 1.5 119.86821677796956


In [4]:
pt_dirs = [#'samplings/DiT/ablations/clip/cfg1.5_s3_N50000/clip_0',
           #'samplings/DiT/ablations/clip/cfg1.5_s5_N50000/clip_0',
           #'samplings/DiT/ablations/clip/cfg1.5_s7_N50000/clip_0',
           'samplings/DiT/ablations/clip/cfg1.5_s9_N50000/clip_0',
          ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    #data = torch.load('/dataset/dit/stats.pt')
    data = torch.load("/home/scpark/data/imagenet_feats/train_clean_stats.pt")
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(config.solver, config.NFE, config.CFG, fid)

# 100%|██████████| 50001/50001 [00:13<00:00, 3621.27it/s]
# Dual-Solver 3 1.5 25.466029364205383
# 100%|██████████| 50001/50001 [00:13<00:00, 3616.40it/s]
# Dual-Solver 5 1.5 4.0834374434331835
# 100%|██████████| 50001/50001 [00:13<00:00, 3736.30it/s]
# Dual-Solver 7 1.5 3.5281183876001023
# 100%|██████████| 50001/50001 [00:13<00:00, 3590.27it/s]
# Dual-Solver 9 1.5 3.0731233074014312

100%|██████████| 50001/50001 [00:15<00:00, 3314.97it/s]


Dual-Solver 9 1.5 3.434591758338911


In [6]:
pt_dirs = ['samplings/DiT/ablations/clip_h/cfg1.5_s3_N50000/clip_h_0',
           'samplings/DiT/ablations/clip_h/cfg1.5_s5_N50000/clip_h_0',
           'samplings/DiT/ablations/clip_h/cfg1.5_s7_N50000/clip_h_0',
           'samplings/DiT/ablations/clip_h/cfg1.5_s9_N50000/clip_h_0',
          ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(config.solver, config.NFE, config.CFG, fid)


  0%|          | 0/50001 [00:00<?, ?it/s]

 12%|█▏        | 6029/50001 [00:02<00:18, 2330.83it/s]


KeyboardInterrupt: 

In [7]:
pt_dirs = ['samplings/DiT/ablations/clip_h/cfg1.5_s5_N50000/clip_h_0',

          ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(config.solver, config.NFE, config.CFG, fid)


  0%|          | 0/50001 [00:00<?, ?it/s]

100%|██████████| 50001/50001 [00:13<00:00, 3596.00it/s]


Dual-Solver 5 1.5 4.834150232741649


In [3]:
pt_dirs = ['samplings/DiT/ablations/multi/cfg1.5_s3_N50000/multi_0',
           'samplings/DiT/ablations/multi/cfg1.5_s5_N50000/multi_0',
           'samplings/DiT/ablations/multi/cfg1.5_s7_N50000/multi_0',
           'samplings/DiT/ablations/multi/cfg1.5_s9_N50000/multi_0',
          ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(config.solver, config.NFE, config.CFG, fid)


  0%|          | 0/50001 [00:00<?, ?it/s]

100%|██████████| 50001/50001 [00:23<00:00, 2086.35it/s]


Dual-Solver 3 1.5 24.550950308189783


100%|██████████| 50001/50001 [00:21<00:00, 2354.31it/s]


Dual-Solver 5 1.5 4.132725043508003


FileNotFoundError: [Errno 2] No such file or directory: 'samplings/DiT/ablations/multi/cfg1.5_s7_N50000/multi_0/config.json'

In [ ]:
pt_dirs = ['samplings/DiT/ablations/multi3/cfg1.5_s3_N50000/multi3_0',
           'samplings/DiT/ablations/multi3/cfg1.5_s5_N50000/multi3_0',
           'samplings/DiT/ablations/multi3/cfg1.5_s7_N50000/multi3_0',
          ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(config.solver, config.NFE, config.CFG, fid)


  5%|▍         | 2326/50001 [00:01<00:20, 2304.87it/s]

In [ ]:
pt_dirs = ['samplings/DiT/ablations/multi3/cfg1.375_s9_N50000/multi3_0',
          ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'], key='inception_feature')
    print(config.solver, config.NFE, config.CFG, fid)

    #3.21

100%|██████████| 5000/5000 [00:01<00:00, 3512.17it/s]


Dual-Solver 9 1.375 9.713832266889199


In [55]:
pt_dirs = ['samplings/DiT/ablations/multi3/cfg1.5_rgb_s9_N50000/multi3_0',
          ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'], key='inception_feature')
    print(config.solver, config.NFE, config.CFG, fid)

    #3.21

100%|██████████| 50001/50001 [00:20<00:00, 2471.36it/s]


Dual-Solver 9 1.5 4.942010796389923


In [63]:
pt_dirs = ['samplings/DiT/1.5_rgb/250/DDPM-Solver/50000/ddpm_1',
          ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'], key='inception_feature')
    print(config.solver, config.NFE, config.CFG, fid)

    #3.21

100%|██████████| 201/201 [00:00<00:00, 2523.61it/s]


DDPM-Solver 250 1.5 146.99129688299098


In [ ]:
pt_dirs = ['samplings/DiT/1.5/3/DS-Solver_DDPM/50000/ds_0',
           'samplings/DiT/1.5/5/DS-Solver_DDPM/50000/ds_0',
           'samplings/DiT/1.5/7/DS-Solver_DDPM/50000/ds_0',
           'samplings/DiT/1.5/9/DS-Solver_DDPM/50000/ds_0',
            ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(config.solver, config.NFE, config.CFG, fid)


100%|██████████| 50001/50001 [00:20<00:00, 2396.75it/s]


DS-Solver_DDPM 3 1.5 67.31452106293881


100%|██████████| 50001/50001 [00:19<00:00, 2619.73it/s]


DS-Solver_DDPM 5 1.5 7.669618176284644


100%|██████████| 50001/50001 [00:19<00:00, 2565.55it/s]


DS-Solver_DDPM 7 1.5 3.7956684458916357


100%|██████████| 50001/50001 [00:18<00:00, 2667.89it/s]


DS-Solver_DDPM 9 1.5 3.0241632564564043


In [ ]:
pt_dirs = ['samplings/DiT/ablations/multi3/cfg1.5_s9_N50000/multi3_4',   
          ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    #data = torch.load('/dataset/dit/valid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(config.solver, config.NFE, config.CFG, fid)
# TF-Evaluation FID: 2.83684451425853

  0%|          | 0/50001 [00:00<?, ?it/s]

100%|██████████| 50001/50001 [00:13<00:00, 3770.05it/s]


Dual-Solver 9 1.5 2.8356757258998186


In [35]:
pt_dirs = ['samplings/dit256_dpms10_inception',
          ]
for pt_dir in pt_dirs:
    #config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(fid)
    #print(config.solver, config.NFE, config.CFG, fid)


100%|██████████| 50000/50000 [00:14<00:00, 3538.28it/s]


4.370393280855865


In [5]:
!ls /dataset/dit/

stats.pt	   train1.5_1k_traj  valid1.5_100.zip
train1.5_10k_traj  train1.5_1k.zip   valid_stats.npz
train1.5_1k	   valid1.5_100      VIRTUAL_imagenet256_labeled.npz
